# Evaluate BLEU / WER / CER

Install dependencies from the repo root (`pip install -r requirements.txt`). Optional: `pip install -e .` so imports work regardless of the notebook working directory.

In [1]:
from pathlib import Path
import sys

_root = Path.cwd().resolve()
if not (_root / "nmt").is_dir() and (_root.parent / "nmt").is_dir():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import os
os.chdir(_root)

import torch
import sacrebleu
from tqdm.auto import tqdm
import torchmetrics
from nmt.config import get_config, get_weights_path
from nmt.checkpoint import load_training_checkpoint
from nmt.train import get_model, get_dataset, greedy_decode

e:\transformer-translation-from-scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = get_config()
train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_dataset(config)
model = get_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)
model_filename = get_weights_path(config, str(config["checkpoint_epoch"]))
state = load_training_checkpoint(model_filename, map_location=device)
model.load_state_dict(state["model_state_dict"])
model.eval()

Max source length: 479, Max target length: 466


Transformer(
  (encoder): Encoder(
    (layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (self_attention_block): MultiHeadAttentionBlock(
          (w_q): Linear(in_features=512, out_features=512, bias=False)
          (w_k): Linear(in_features=512, out_features=512, bias=False)
          (w_v): Linear(in_features=512, out_features=512, bias=False)
          (w_o): Linear(in_features=512, out_features=512, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (feed_forward_block): FeedForwardBlock(
          (linear_1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear_2): Linear(in_features=2048, out_features=512, bias=True)
        )
        (residual_connections): ModuleList(
          (0-1): 2 x ResidualConnection(
            (dropout): Dropout(p=0.1, inplace=False)
            (norm): LayerNormalization()
          )
        )
      )
    )
    (norm): LayerNormalization

In [5]:
def compute_bleu(model, dataloader, tokenizer_src, tokenizer_tgt, config, device, num_batches=100):
    references = []
    hypotheses = []
    wer = torchmetrics.text.WordErrorRate()
    cer = torchmetrics.text.CharErrorRate()

    with torch.no_grad():
        for i, batch in enumerate(tqdm(dataloader, desc="Computing metrics", total=num_batches)):
            if i >= num_batches:
                break
            encoder_input = batch["encoder_input"].to(device)
            encoder_mask = batch["encoder_mask"].to(device)
            tgt_text = batch["tgt_text"][0]
            model_output = greedy_decode(
                model,
                encoder_input,
                encoder_mask,
                tokenizer_src,
                tokenizer_tgt,
                config["seq_len"],
                device,
            )
            pred = tokenizer_tgt.decode(model_output.cpu().numpy())
            references.append([tgt_text])
            hypotheses.append(pred)
            wer.update(pred, tgt_text)
            cer.update(pred, tgt_text)

    bleu = sacrebleu.corpus_bleu(hypotheses, list(zip(*references)))
    print(f"BLEU score: {bleu.score:.2f}")
    print(f"WER: {wer.compute():.4f}")
    print(f"CER: {cer.compute():.4f}")
    return bleu, wer.compute(), cer.compute()

In [6]:
compute_bleu(model, val_dataloader, tokenizer_src, tokenizer_tgt, config, device, num_batches=100)

Computing metrics: 100%|██████████| 100/100 [01:50<00:00,  1.10s/it]

BLEU score: 52.27
WER: 0.7129
CER: 0.3521


(BLEU = 52.27 72.9/55.3/46.2/40.1 (BP = 1.000 ratio = 1.002 hyp_len = 2196 ref_len = 2192),
 tensor(0.7129),
 tensor(0.3521))

In [7]:
# TensorBoard (log dir is under the repo root: runs/)
%load_ext tensorboard
%tensorboard --logdir=runs